# Latin Lemmatizer - Basic Usage Examples

This notebook demonstrates how to use the `latin_lemmatizer` package to query Latin lemmas and inflected forms from the database.

## Setup

**Before running this notebook**, make sure you have:
1. Run the installation cell below to install psycopg
2. Set the `DATABASE_URL` environment variable
3. Populated the database with Latin data

In [ ]:
# Install dependencies if not already installed
import subprocess
import sys

try:
    import psycopg
    print("✓ psycopg already installed")
except ImportError:
    print("Installing psycopg[binary]...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "psycopg[binary]"])
    print("✓ Installation complete!")

## Import the Package

Now we can import the latin_lemmatizer package.

In [ ]:
import sys
from pathlib import Path

# Add parent directory to path so we can import latin_lemmatizer
sys.path.insert(0, str(Path().resolve().parent))

from latin_lemmatizer import get_lemma, get_form, LatinLemmatizer

print("✓ Successfully imported latin_lemmatizer")

## Example 1: Get Lemma from Any Word

The `get_lemma()` function can find the lemma (dictionary headword) from either:
- A lemma itself (returns that lemma)
- An inflected form (returns its lemma)

In [ ]:
# Try different words - some are lemmas, some are inflected forms
words = ["amavi", "amat", "amo", "rosam", "rosa"]

for word in words:
    lemma = get_lemma(word)
    if lemma:
        print(f"{word:15} → {lemma['lemma_diac']:15} ({lemma['pos']})")
    else:
        print(f"{word:15} → Not found")

## Example 2: Get All Forms of a Lemma

Use `get_form(lemma="...")` to retrieve all inflected forms of a specific lemma.

In [ ]:
# Get all forms of "amo"
forms = get_form(lemma="amo")
print(f"Found {len(forms)} forms of 'amo'\n")

# Show first 10 forms with their morphological features
print(f"{'Form':<20} {'Mood':<12} {'Tense':<15} {'Voice':<8} {'Person':<6} {'Number'}")
print("-" * 80)
for f in forms[:10]:
    print(f"{f['form_diac']:<20} {f['mood'] or '-':<12} {f['tense'] or '-':<15} {f['voice'] or '-':<8} {f['person'] or '-':<6} {f['number'] or '-'}")

if len(forms) > 10:
    print(f"\n... and {len(forms) - 10} more forms")

## Example 3: Get Specific Forms with Filters

You can filter forms by morphological features like mood, tense, voice, person, number, etc.

In [ ]:
# Get present indicative active forms of "amo"
forms = get_form(
    lemma="amo",
    mood="indicative",
    tense="present",
    voice="active"
)

print("Present Indicative Active Forms of 'amo':")
for f in forms:
    print(f"  {f['form_diac']:15} ({f['person']} {f['number']})")

## Example 4: Inflect from an Existing Form

Instead of starting with a lemma, you can start with any inflected form and find other forms of the same lemma.

In [ ]:
# Get plural forms from the same lemma as "amavi"
forms = get_form(form="amavi", number="plural")

print(f"Found {len(forms)} plural forms from the same lemma as 'amavi':\n")
print(f"{'Form':<20} {'Tense':<15} {'Voice':<8} {'Person'}")
print("-" * 60)
for f in forms[:10]:
    print(f"{f['form_diac']:<20} {f['tense'] or '-':<15} {f['voice'] or '-':<8} {f['person'] or '-'}")

if len(forms) > 10:
    print(f"\n... and {len(forms) - 10} more")

## Example 5: Get Verb Forms (Infinitive, Participle, etc.)

Use the `verb_form` parameter to get specific verb forms like infinitive, participle, gerund, gerundive, or supine.

In [ ]:
# Get infinitive forms of "amo"
forms = get_form(lemma="amo", verb_form="infinitive")

print("Infinitive forms of 'amo':")
for f in forms:
    print(f"  {f['form_diac']:15} (voice: {f['voice'] or 'active'})")

## Example 6: Using the Client Class

For more control, you can use the `LatinLemmatizer` client class directly, especially useful with context managers for proper connection handling.

In [ ]:
# Using the client class with context manager
with LatinLemmatizer() as client:
    # Get lemma
    lemma = client.get_lemma("rosam")
    print(f"Lemma of 'rosam': {lemma['lemma_diac'] if lemma else 'Not found'}")
    
    # Get accusative forms
    if lemma:
        acc_forms = client.get_form(lemma=lemma['lemma_nod'], case="accusative")
        print(f"\nAccusative forms: {', '.join(f['form_diac'] for f in acc_forms)}")

## Example 7: More Complex Queries

Try combining multiple filters to find very specific forms.

In [ ]:
# Example: Get perfect passive participle forms
forms = get_form(lemma="amo", verb_form="participle", voice="passive", tense="perfect")

print("Perfect Passive Participles of 'amo':")
for f in forms:
    print(f"  {f['form_diac']:20} ({f['gender']} {f['number']} {f['case']})")

## Example 8: Explore Noun Declensions

For nouns, you can filter by case, number, and gender.

In [ ]:
# Get all forms of "rosa" (a noun)
forms = get_form(lemma="rosa")

print(f"Found {len(forms)} forms of 'rosa'\n")
print("All cases and numbers:")
print(f"{'Form':<15} {'Case':<12} {'Number':<10} {'Gender'}")
print("-" * 50)
for f in forms:
    print(f"{f['form_diac']:<15} {f['case'] or '-':<12} {f['number'] or '-':<10} {f['gender'] or '-'}")

# Get just genitive singular
gen_sg = get_form(lemma="rosa", case="genitive", number="singular")
print(f"\nGenitive singular: {gen_sg[0]['form_diac'] if gen_sg else 'Not found'}")

## Tips

- Always use `lemma` OR `form` (never both) in `get_form()`
- All filter parameters are optional - omit them to get all matching forms
- The `form_nod` field contains normalized text (no diacritics), while `form_diac` has diacritics
- Use the client class for better connection management in long-running scripts